# 03 Feature engineering

**Owner:** Person B. Start once `outputs/column_roles.json` exists. The full frame comes from Person A.

**Inputs:** `data/processed/analysis_frame.parquet` and `outputs/column_roles.json`.

**Outputs:** `data/processed/features.parquet`, `outputs/feature_log.json`, `outputs/tables/two_way.csv`, `outputs/figures/pca_descriptive.png`.

**Stop rule:** PCA and LDA in this notebook are descriptive and use the full analysis frame. They are not model features. Train-only aggregates are built in notebook 04 after the frozen split. If the questions are descriptive, stop after the two-way table and the PCA plot.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.paths import (
    RAW,
    PROCESSED,
    OUTPUTS,
    TABLES,
    ANALYSIS_PATH,
    FEATURES_PATH,
    MODEL_MATRIX_PATH,
    ROLES_PATH,
    CONTRACT_PATH,
    GATE_PATH,
    FEATURE_LOG_PATH,
    ALIAS_PATH,
    ANSWERS_PATH,
    COMPARISON_PATH,
    BUNDLE_PATH,
    ensure_dirs,
)

ensure_dirs()
print("root", ROOT)


In [ ]:
import json

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd

from src import io as sio
from src.features import add_time_parts, drop_low_information, id_from_descriptors, suggest_two_way_columns, two_way
from src.reduce import run_lda, run_pca
from src.roles import columns_with_role, read_roles

if not ANALYSIS_PATH.exists() and not ANALYSIS_PATH.with_suffix(".csv").exists():
    print("STOP: analysis frame is not written yet. Person A owns notebook 02.")
else:
    frame = sio.read_frame(ANALYSIS_PATH).reset_index(drop=True)
    roles = read_roles()
    protect = [
        column
        for column in frame.columns
        if column
        in set(
            columns_with_role(roles, "id")
            + columns_with_role(roles, "time")
            + columns_with_role(roles, "geo")
            + columns_with_role(roles, "target")
        )
    ]
    frame, dropped = drop_low_information(frame, protect=protect)
    print("dropped", dropped.to_dict(orient="records"))
    descriptors = [column for column in columns_with_role(roles, "descriptor") if column in frame.columns]
    if descriptors:
        frame["entity_from_descriptors"] = id_from_descriptors(frame, descriptors)
        print("descriptor-key cardinality", int(frame["entity_from_descriptors"].nunique(dropna=True)))
    time_columns = [column for column in columns_with_role(roles, "time") if column in frame.columns]
    if time_columns:
        frame = pd.concat([frame, add_time_parts(frame, time_columns[0])], axis=1)
    target = next((column for column in columns_with_role(roles, "target") if column in frame.columns), None)
    row_column, col_column = suggest_two_way_columns(frame, roles)
    log = {
        "dropped": dropped.to_dict(orient="records"),
        "descriptor_columns": descriptors,
        "two_way": None,
        "pca": None,
        "lda": None,
        "aggregates": "Train-only aggregates are added in notebook 04. Nothing in this file's group profile is a model column.",
    }
    if row_column and col_column:
        value = target if target and pd.api.types.is_numeric_dtype(frame[target]) else None
        table = two_way(frame, row_column, col_column, target=value)
        table.to_csv(TABLES / "two_way.csv")
        log["two_way"] = {
            "row": row_column,
            "column": col_column,
            "target": value,
            "scope": "descriptive_full_frame",
        }
        print("two-way", row_column, "x", col_column, table.shape)
    else:
        print("No two low-cardinality columns yet. Set descriptor roles or add a time part.")
    measures = [column for column in columns_with_role(roles, "measure") if column in frame.columns]
    pca = run_pca(frame, measures, scope="descriptive_full_frame")
    log["pca"] = {
        "ok": pca["ok"],
        "scope": pca.get("scope"),
        "explained_variance_ratio": pca.get("explained_variance_ratio"),
        "reason": pca.get("reason"),
    }
    if pca["ok"]:
        scores = pca["scores"]
        fig, ax = plt.subplots(figsize=(6, 5))
        vertical = scores[:, 1] if scores.shape[1] > 1 else [0] * len(scores)
        ax.scatter(scores[:, 0], vertical, s=8, alpha=0.45)
        ratio = pca["explained_variance_ratio"]
        ax.set_xlabel(f"PC1 ({ratio[0]:.0%})")
        if len(ratio) > 1:
            ax.set_ylabel(f"PC2 ({ratio[1]:.0%})")
        ax.set_title("PCA on the full analysis frame. Descriptive only, not a model feature.")
        sio.save_figure(fig, "pca_descriptive.png")
        pca["loadings"].to_csv(TABLES / "pca_loadings.csv")
        print("PCA variance", ratio)
    else:
        print("PCA skipped:", pca.get("reason"))
    if target:
        lda = run_lda(frame, measures, target, scope="descriptive_full_frame")
        log["lda"] = {"ok": lda["ok"], "scope": lda.get("scope"), "reason": lda.get("reason")}
        print("LDA", log["lda"])
    else:
        print("LDA skipped: no column is tagged target.")
        log["lda"] = {"ok": False, "reason": "no target role"}
    written = sio.save_frame(frame, FEATURES_PATH)
    FEATURE_LOG_PATH.write_text(json.dumps(log, indent=2), encoding="utf-8")
    print("wrote", written, frame.shape)
